In [1]:
import copy 
import json 
import random 
import time 
from pathlib import Path 
from xgboost import XGBRegressor
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 
import torch 
from torch import nn 
from scipy.stats import norm 
from sklearn.preprocessing import StandardScaler 
from sklearn.dummy import DummyRegressor 
from sklearn.linear_model import LinearRegression 
import joblib 

In [ ]:
FEATURES = ['S0', 'K', 'T', 'r', 'v0', 'kappa', 'theta', 'sigma', 'rho']
TRAIN_N, VALIDATION_N, TEST_N = 12000, 200, 500 
EPOCHS, PATIENCE, BATCH_SIZE, LEARNING_RATE, MIN_DELTA = 1000, 100, 64, 1e-3, 1e-7
MODEL_SEED = (42, 43, 44)
HIDDEN_UNITS = (128, 128, 128)

REUSE_BASELINE = True 
BASELINE_RESULTS_DIR = Path('experiment_baseline')
DATA_DIR = Path('experiment_training_size_results') / 'data'
ARCHITECTURE_DIR = Path('experiment_architecture_results')/f'n{TRAIN_N}'
OUTPUT_DIR = Path('experiment_model_comparison_results') / f'n{TRAIN_N}'
MODEL_DIR = OUTPUT_DIR / 'models'
HISTORY_DIR = OUTPUT_DIR / 'histories'
PREDICTION_DIR = OUTPUT_DIR / 'predictions'
FIGURE_DIR = OUTPUT_DIR / 'figures'

for directory in (MODEL_DIR, HISTORY_DIR, PREDICTION_DIR, FIGURE_DIR): 
    directory.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device('cpu')
#torch.set_num_threads(1)

XGB_PARAMS = dict(
    n_estimators=100, 
    max_depth=5, 
    learning_rate = 0.05, 
    subsample = 0.8, 
    colsample_bytree=0.8, 
    objective="reg:squarederror",
    tree_method="hist", 
    #n_jobs=1
)
print('Training Size: ', TRAIN_N)